In [2]:
from pathlib import Path
import pandas as pd
from credit_card_fraud_detection.paths import ROOT, RAW
from datetime import date

In [3]:
train = RAW / "train.csv"
test = RAW / "test.csv"

In [4]:
train_df = pd.read_csv(train)
test_df = pd.read_csv(test)

### Initial Profiling

Shape and Memory

In [5]:
# shape and memory 
print(train_df.shape)
print(train_df.info(memory_usage="deep"))

(699960, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            699960 non-null  int64  
 1   amount_usd                    699960 non-null  float64
 2   hours_since_last_txn          699960 non-null  float64
 3   txn_count_last_24h            699960 non-null  int64  
 4   distance_from_home_km         699960 non-null  float64
 5   card_age_months               699960 non-null  int64  
 6   customer_age                  699960 non-null  int64  
 7   account_balance_usd           699960 non-null  float64
 8   cvv_retry_count               699960 non-null  int64  
 9   velocity_score                699960 non-null  float64
 10  time_of_day_hour              699960 non-null  int64  
 11  day_of_week                   699960 non-null  int64  
 12  merchant_risk_score           6

Head, tail and random sample

In [9]:
train_df.head(); 
train_df.tail()
train_df.sample(10, random_state=0)

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
549946,549946,16.43,3.11,3,8.04,53,23,1324.51,0,17.4,...,PIN,Contactless,Smart Watch,False,False,False,False,False,False,0.0
151381,151381,42.30,5.20,3,25.37,42,31,489.98,0,6.5,...,OTP,Online,Android Phone,False,False,False,False,False,False,0.0
82749,82749,141.49,5.61,5,4.10,48,21,1253.05,0,28.4,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
547892,547892,11.23,8.80,4,6.71,45,67,692.45,0,30.2,...,OTP,Contactless,Android Phone,False,False,False,False,False,False,0.0
543808,543808,559.64,6.82,4,0.59,45,43,1744.10,0,22.0,...,PIN,Online,iPhone,False,False,False,False,False,False,0.0
676435,676435,18.59,6.52,6,6.19,44,51,5046.61,0,35.4,...,OTP,In-App,Windows PC,False,True,False,False,False,False,0.0
571871,571871,45.72,5.32,0,12.92,41,18,1267.10,1,5.4,...,PIN,Contactless,Android Phone,False,False,False,False,False,False,0.0
217772,217772,16.35,15.40,1,12.10,49,67,590.66,0,0.0,...,PIN,Online,iPhone,False,False,False,False,False,False,0.0
641564,641564,27.64,2.15,1,8.32,42,43,1635.27,0,17.0,...,No Authentication,Online,iPhone,False,False,False,False,False,False,0.0
220833,220833,7.55,25.59,1,0.93,49,74,2350.08,0,0.0,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


Data type audit

In [10]:
train_df.dtypes

id                                int64
amount_usd                      float64
hours_since_last_txn            float64
txn_count_last_24h                int64
distance_from_home_km           float64
card_age_months                   int64
customer_age                      int64
account_balance_usd             float64
cvv_retry_count                   int64
velocity_score                  float64
time_of_day_hour                  int64
day_of_week                       int64
merchant_risk_score             float64
prior_disputes                    int64
merchant_category                object
card_type                        object
auth_method                      object
channel                          object
device_type                      object
is_foreign_transaction             bool
is_new_merchant                    bool
used_vpn                           bool
ip_country_mismatch                bool
billing_shipping_mismatch          bool
is_ai_generated_scam_attempt       bool


In [11]:
train_df.select_dtypes("object").head()

,merchant_category,card_type,auth_method,channel,device_type
0,Groceries,Mastercard,OTP,POS,iPhone
1,Online Retail,Visa,PIN,In-App,Android Phone
2,Utilities,Visa,OTP,Online,Tablet
3,Groceries,Mastercard,3D Secure,POS,Android Phone
4,Electronics,Visa,3D Secure,Online,Windows PC


Missing-value summary

In [12]:
train_df.isna().mean().sort_values(ascending=False)

id                              0.0
amount_usd                      0.0
is_ai_generated_scam_attempt    0.0
billing_shipping_mismatch       0.0
ip_country_mismatch             0.0
used_vpn                        0.0
is_new_merchant                 0.0
is_foreign_transaction          0.0
device_type                     0.0
channel                         0.0
auth_method                     0.0
card_type                       0.0
merchant_category               0.0
prior_disputes                  0.0
merchant_risk_score             0.0
day_of_week                     0.0
time_of_day_hour                0.0
velocity_score                  0.0
cvv_retry_count                 0.0
account_balance_usd             0.0
customer_age                    0.0
card_age_months                 0.0
distance_from_home_km           0.0
txn_count_last_24h              0.0
hours_since_last_txn            0.0
is_fraud                        0.0
dtype: float64

In [13]:
train_df.isna().any(axis=1).mean()

0.0

Cardinality (Unique counts) per column

In [14]:
train_df.nunique().sort_values()

is_fraud                             2
is_ai_generated_scam_attempt         2
is_new_merchant                      2
used_vpn                             2
is_foreign_transaction               2
ip_country_mismatch                  2
billing_shipping_mismatch            2
cvv_retry_count                      4
prior_disputes                       5
card_type                            5
auth_method                          5
channel                              5
day_of_week                          7
device_type                          8
merchant_category                   12
txn_count_last_24h                  12
time_of_day_hour                    24
card_age_months                     49
customer_age                        64
velocity_score                     666
merchant_risk_score                950
hours_since_last_txn              3633
distance_from_home_km             4365
amount_usd                       13387
account_balance_usd              43317
id                       

Summary Statistics

In [15]:
train_df.describe(percentiles=[.01, .05, .5, .95, .99]).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
id,699960.0,349979.500000,202061.191548,0.00,6999.59,34997.95,349979.50,664961.05,692959.41,699959.00
amount_usd,699960.0,79.893068,205.173538,1.00,4.42,6.86,27.25,542.46,936.95,6872.69
hours_since_last_txn,699960.0,6.633734,6.290133,0.01,0.11,0.47,4.73,19.40,29.38,51.98
txn_count_last_24h,699960.0,2.984092,1.498998,0.00,0.00,1.00,3.00,6.00,7.00,11.00
distance_from_home_km,699960.0,13.587575,11.617325,0.00,0.08,0.76,10.22,37.94,47.86,57.93
card_age_months,699960.0,46.862982,5.579190,25.00,34.00,38.00,47.00,56.00,60.00,73.00
customer_age,699960.0,51.430286,17.899150,18.00,19.00,21.00,52.00,79.00,81.00,81.00
account_balance_usd,699960.0,2064.998070,3400.132922,58.25,454.85,532.61,1382.74,4861.18,9082.81,99701.39
cvv_retry_count,699960.0,0.085472,0.289524,0.00,0.00,0.00,0.00,1.00,1.00,3.00
velocity_score,699960.0,18.929445,10.864385,0.00,0.00,0.00,18.60,37.40,46.10,72.90


Value counts for categorical columns (top and bottom)

In [17]:
for c in train_df.select_dtypes("object"):
    print(train_df[c].value_counts(dropna=False).head(10))
    print(train_df[c].value_counts().tail(10))

merchant_category
Online Retail    155097
Groceries        142578
Restaurants      101360
Fuel              59880
Electronics       48693
Streaming         47196
Utilities         42027
Healthcare        35889
Travel            28715
Gaming            23784
Name: count, dtype: int64
merchant_category
Restaurants        101360
Fuel                59880
Electronics         48693
Streaming           47196
Utilities           42027
Healthcare          35889
Travel              28715
Gaming              23784
Gift Cards           9456
Crypto Exchange      5285
Name: count, dtype: int64
card_type
Visa          345001
Mastercard    254857
Amex           45797
RuPay          37070
Discover       17235
Name: count, dtype: int64
card_type
Visa          345001
Mastercard    254857
Amex           45797
RuPay          37070
Discover       17235
Name: count, dtype: int64
auth_method
3D Secure            210036
OTP                  175178
PIN                  154020
Biometric             93666
No Aut

Duplicate count

In [18]:
train_df.duplicated().sum()

0

In [19]:
train_df.duplicated(subset=["id"]).sum()

0